# Lab 06 · Image and NLP Enrichment (Notebook Walkthrough)

**Concept.** The `visual_nlp` profile adds `OCRSkill`, `ImageAnalysisSkill`, and `LanguageDetectionSkill`. This matters for a diagram-heavy engineering document where evidence lives in figures, not just prose.

**Critical nuance (which fields reach which retrieval mode):**
- **Image descriptions** (`image_description_text`) are merged back onto the canonical chunks → visible to `full_text` / `vector` / `hybrid`.
- **OCR text** and **detected language** live only in the Search-managed enrichment index → surfaced by **Agentic** retrieval.

So Hybrid reflects image descriptions, while Agentic additionally brings in OCR text the direct modes cannot reach.

## Step 1 — Bootstrap

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
sys.path.insert(0, str(NB_DIR if (NB_DIR / 'lab_runtime.py').exists() else NB_DIR / 'notebooks'))
import lab_runtime as lab

info = lab.bootstrap()
info

{'repo_root': 'C:\\Users\\samsonlee\\rag-on-azure-lab',
 'env_values_loaded': 89,
 'search_endpoint': 'https://your-search-service.search.windows.net',
 'search_configured': True,
 'embedding_deployment': 'text-embedding-3-large-vector',
 'chat_deployment': 'gpt-5-4-mini-chat',
 'agentic_planning_model': 'gpt-5-4-mini-search',
 'canonical_index': 'ai-search-lab-index'}

## Step 2 — Ingest with image + NLP enrichment

In [2]:
job = lab.ingest(skill_profile='visual_nlp', reuse=True)
lab.chunk_overview(job)

Reusing existing 'visual_nlp' ingestion (doc_id=edbcb680, 414 chunks). Pass reuse=False to force a fresh run.


{'doc_id': 'edbcb680033d4886af1b6cfab59ccddc',
 'skill_profile': 'visual_nlp',
 'chunk_count': 414,
 'avg_tokens': 208.4,
 'max_tokens': 1976,
 'chunks_with_summary': 414,
 'chunks_with_keyword_hints': 414,
 'chunks_with_image_description': 0}

`chunks_with_image_description` should now be **non-zero** — image analysis descriptions were merged onto the canonical chunks.

## Step 3 — Inspect the image descriptions merged onto chunks

In [3]:
import pandas as pd

chunks = lab.load_chunks(job)
with_img = [c for c in chunks if c.get('image_description_text')]
print('chunks carrying image descriptions:', len(with_img))
pd.DataFrame([
    {
        'section': ' > '.join(c.get('section_path') or []) or '(root)',
        'pages': c.get('page_numbers'),
        'image_description': (c.get('image_description_text') or '')[:180],
    }
    for c in with_img[:6]
])

chunks carrying image descriptions: 0


""


## Step 4 — Hybrid vs Agentic on a visual question

Hybrid reflects the merged image descriptions; Agentic additionally reaches the OCR text in the enrichment index.

In [4]:
QV = 'What do the figures on ground settlement and wall deflection show, and what visual evidence supports the answer?'

print('--- HYBRID (sees image descriptions) ---')
resp_hybrid = lab.ask(QV, job=job, retrieval_mode='hybrid', record_as='lab06_visual_hybrid')
lab.show_answer(resp_hybrid, max_citations=4)

--- HYBRID (sees image descriptions) ---


[retrieval_mode=hybrid | scoring_profile=default | citations=7]

However, this method only calculates the lateral deflection of the embedded wall and estimation of ground settlement is usually based on empirical correlations. Lui & Yau (1995) reported the back analysis of the basement excavation at the Dragon Centre, Kowloon. [1]

Supporting evidence:
- 8v / He (%) ôv = 0.50 8h 8h / Hẹ (%) where 8h = Maximum lateral wall deflection He = Maximum excavation depth Figure 8.4 Relationship between Ground Settlement and Lateral Wall Deflection from Case Study Data Consolidation settlements are seldom estimated using empirical methods. Instead, they are commonly assessed by carrying out one-dimensional consolidati [7]
- Clough & O'Rourke (1990) discussed typical profiles of wall deflection and the adjacent ground deformation based on case histories (Figure 8.3). During the initial stage, dewatering and soil excavation are carried out before installation of the first lateral support. [2]

Cita

In [5]:
print('--- AGENTIC (adds OCR from enrichment index) ---')
resp_agentic = lab.ask(QV, job=job, retrieval_mode='agentic', record_as='lab06_visual_agentic')
lab.show_answer(resp_agentic, max_citations=6)

--- AGENTIC (adds OCR from enrichment index) ---


[retrieval_mode=agentic | scoring_profile=default | citations=8]

- The figures show that **ground settlement and wall deflection are related, but the relationship is not always exact**. In the case-study data, **small deformations show no apparent correlation**, while for larger cases the **maximum ground settlement is usually about 50% of the maximum lateral wall deflection**, with a noted exception for Tsuen Wan West Station, where the settlement was about **0.75 to 1.0 times** the wall deflection. [1][2][3]
- The figures also show that **ground settlement generally increases with excavation depth**, but there is **substantial scatter** in the data, meaning different projects can behave quite differently. [4][5]
- The visual evidence supporting this is **Figure 8.4**, which is explicitly described as **“Relationship between Ground Settlement and Lateral Wall Deflection from Case Study Data”** and plots **8v/He (%)** against **8h/He (%)**. [3][6]
- Supporting figure evidence also inc

## Takeaways

- Image-analysis descriptions are merged to canonical chunks, so **Hybrid can already reflect visual evidence**.
- OCR text and detected language are enrichment-index-only, so **Agentic surfaces evidence the direct modes cannot**.
- For diagram-heavy documents, compare Hybrid and Agentic to see the full benefit of visual + NLP enrichment.

Next: **Lab 07** focuses on agentic retrieval mechanics (planning, subqueries, grounded synthesis).